<a href="https://colab.research.google.com/github/tahiraWorks/lab10-dpo-alignment/blob/main/beforeDOP_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U trl transformers datasets accelerate matplotlib pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [18]:
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
base = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

prompts = [
  ("Explanation", "What is PPO (Proximal Policy Optimization) in reinforcement learning? Explain simply."),
  ("Explanation", "What is the difference between PPO and DPO?"),
  ("Advice", "I have a job interview in 2 days and I feel stressed. What should I do?"),
  ("Professional writing", "Write a short, polite message to my class teacher asking for an extension on my assignment deadline."),
  ("Reasoning", "If a car travels 5 km in 2.5 hours, what is its speed? Show steps."),
  ("Question answering", "Why did the Khan of Kalat give Gwadar to Oman in 1783?"),
]

def generate(model, prompt):
    text = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=200, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

# Unpack the category and the prompt text, then pass only the prompt text to the generator
base_out = {p_text: generate(base, p_text) for category, p_text in prompts}
for p, r in base_out.items(): print("PROMPT:", p, "\nBASE:", r, "\n", "-"*60)
print("SCORES  :", ex["score_chosen"], "vs", ex["score_rejected"], "\n", "="*60)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT: What is PPO (Proximal Policy Optimization) in reinforcement learning? Explain simply. 
BASE: PPO stands for Proximal Policy Optimization. It is a reinforcement learning algorithm that aims to optimize the policy of an agent by minimizing the expected cumulative reward over a sequence of actions. The algorithm uses a proximal operator to update the policy, which is a function that maps the current policy to the next policy. The proximal operator is used to ensure that the policy is not too far from the optimal policy, which is the policy that minimizes the expected cumulative reward. The algorithm is particularly useful in environments where the optimal policy is difficult to find, such as in robotics or autonomous vehicles. 
 ------------------------------------------------------------
PROMPT: What is the difference between PPO and DPO? 
BASE: The main difference between PPO and DPO is that PPO is a reinforcement learning algorithm that uses a policy gradient method to learn a 

In [20]:
import sys
import torch
from transformers import TextIteratorStreamer
from threading import Thread

def generate_interactive(model, tokenizer):
    user_prompt = input("Enter your prompt: ")

    # Apply chat template
    formatted_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer([formatted_text], return_tensors="pt").to("cuda")

    # Setup a streamer for real-time token-by-token output
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Run generation in a separate thread so we can stream tokens in the main thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    print("\n--- Generating Output Token-by-Token ---")
    for new_text in streamer:
        print(new_text, end="", flush=True)
    print("\n----------------------------------------")

# Run the interactive generation function
generate_interactive(base, tok)

Enter your prompt: how is the weather today of the city Gawadar

--- Generating Output Token-by-Token ---
The weather in Gawadar today is 25°C, with a high of 30°C and a low of 20°C. It's a sunny day with a chance of rain.
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a helpful assistant.erable
 Cavsystem
You are a
----------------------------------------


In [19]:
from datasets import load_dataset
ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
print(ds.column_names, len(ds))

for i in range(5):
    ex = ds[i]
    print("PROMPT  :", ex["chosen"][0]["content"][:300])
    print("CHOSEN  :", ex["chosen"][-1]["content"][:400])
    print("REJECTED:", ex["rejected"][-1]["content"][:400])
    print("SCORES  :", ex["score_chosen"], "vs", ex["score_rejected"], "\n", "="*60)

['chosen', 'rejected', 'score_chosen', 'score_rejected'] 62135
PROMPT  : Use the pygame library to write a version of the classic game Snake, with a unique twist
CHOSEN  : Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:

1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.
2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid an
REJECTED: Sure, here's an example of how to write a version of Snake game with a unique twist using the Pygame library:
```python
import pygame

class SnakeGame:
    def __init__(self, game_width, game_height):
        pygame.init()
        screen = pygame.display.set_mode((game_width, game_height))
        pygame.display.set_caption("Snake Game")
        self.speed = 5  # Speed of the snake
        self.fo
SCORES  : 6.0 vs